<a href="https://colab.research.google.com/github/GarimaChaubey/NLP-Natural-Language-Preprocessing-/blob/main/NER_Theory_to_Complete_Implementation_Classroom.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Named Entity Recognition (NER): Theory to Complete Implementation


This notebook teaches NER from fundamentals to implementation using **spaCy**:
1. Understand entities, labels, tokenization, BIO tagging, and the mathematical formulation.
2. Run a pretrained NER model.
3. Train a small custom NER model on annotated examples.
4. Evaluate entity-level precision, recall, and F1.
5. Save and reload the model.

Run in Google Colab or Jupyter. The custom dataset is synthetic and intended only to demonstrate the training workflow.

## 1. What is Named Entity Recognition?

**Named Entity Recognition (NER)** is an NLP information-extraction task. It finds spans of text that refer to named entities and assigns each span a category.

Example: `Sundar Pichai works at Google in California.`

| Text span | Label | Meaning |
|---|---|---|
| Sundar Pichai | PERSON | Person |
| Google | ORG | Organization |
| California | GPE | Geopolitical entity |

NER answers two questions: **where is the entity span?** and **what type is it?**

NER is different from sentiment analysis. Sentiment analysis predicts opinion or emotion; NER extracts entities.

## 2. Common entity categories

| Label | Meaning | Example |
|---|---|---|
| PERSON | Person's name | Marie Curie |
| ORG | Organization | Google, WHO |
| GPE | Country, city, state | India, Paris |
| LOC | Other location | Sahara Desert |
| DATE | Date or period | 20 November 2026 |
| TIME | Time | 9:30 AM |
| MONEY | Monetary value | $50 |
| PERCENT | Percentage | 25% |
| PRODUCT | Product | iPhone |
| EVENT | Named event | Olympic Games |

The exact labels depend on the dataset. One dataset may use `LOCATION`, while another distinguishes `GPE` and `LOC`.

## 3. NER as sequence labeling: BIO tagging

Sentence: `Barack Obama visited New Delhi`

Tokens: `Barack`, `Obama`, `visited`, `New`, `Delhi`

| Token | BIO tag | Meaning |
|---|---|---|
| Barack | B-PERSON | Beginning of a person entity |
| Obama | I-PERSON | Inside the same person entity |
| visited | O | Outside an entity |
| New | B-GPE | Beginning of a GPE entity |
| Delhi | I-GPE | Inside the same GPE entity |

- **B-X:** beginning of an entity of type X.
- **I-X:** inside an entity of type X.
- **O:** outside any entity.

Another representation uses character offsets. For `Barack Obama`, start=0 and end=12. Python uses an inclusive start and exclusive end, so `text[0:12]` returns the entity.

## 5. Applications

- Healthcare: diseases, medicines, symptoms, procedures, hospitals.
- Finance: companies, currencies, monetary values, dates.
- Legal NLP: people, courts, organizations, dates.
- Search and question answering: people, places, organizations.
- News analysis: people, organizations, locations, events.

A general English NER model may not recognize specialized clinical or legal terms reliably. Domain-specific annotation and evaluation are needed.

## 6. Install spaCy

In [ ]:
%pip -q install -U spacy pandas scikit-learn

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


Download a small English pretrained pipeline.

In [ ]:
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 25.1 MB/s  0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## 7. Load the pretrained model
A spaCy pipeline is a sequence of processing components, including a named entity recognizer.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")
print("Pipeline components:", nlp.pipe_names)
print("NER labels:", nlp.get_pipe("ner").labels)

Pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']
NER labels: ('CARDINAL', 'DATE', 'EVENT', 'FAC', 'GPE', 'LANGUAGE', 'LAW', 'LOC', 'MONEY', 'NORP', 'ORDINAL', 'ORG', 'PERCENT', 'PERSON', 'PRODUCT', 'QUANTITY', 'TIME', 'WORK_OF_ART')


## 8. Run NER on one sentence

In [ ]:
text = "Sundar Pichai works at Google in California."
doc = nlp(text)

print("Original text:", text)
for ent in doc.ents:
    print("Text:", ent.text,
          "| Label:", ent.label_,
          "| Start:", ent.start_char,
          "| End:", ent.end_char)

Original text: Sundar Pichai works at Google in California.
Text: Sundar Pichai | Label: PERSON | Start: 0 | End: 13
Text: Google | Label: ORG | Start: 23 | End: 29
Text: California | Label: GPE | Start: 33 | End: 43


### Understand the output
- `ent.text`: extracted entity text.
- `ent.label_`: human-readable entity category.
- `ent.start_char`: starting character index.
- `ent.end_char`: ending index, excluded from the span.

Check that the offsets recover the exact text.

In [ ]:
for ent in doc.ents:
    recovered = text[ent.start_char:ent.end_char]
    print(ent.text, "->", recovered, "| correct:", recovered == ent.text)

Sundar Pichai -> Sundar Pichai | correct: True
Google -> Google | correct: True
California -> California | correct: True


## 9. Visualize entities
The `displacy` renderer highlights the entities and labels.

In [ ]:
from spacy import displacy
displacy.render(doc, style="ent", jupyter=True)

## 10. Try multiple examples
Pretrained models can miss entities or assign unexpected labels. Inspect predictions rather than assuming they are always correct.

In [ ]:
examples = [
    "Microsoft was founded by Bill Gates and Paul Allen.",
    "The World Health Organization is based in Geneva.",
    "Riya moved from Mumbai to London in June 2024.",
    "The meeting starts at 9:30 AM on Monday.",
    "Tesla announced a new product in Germany."
]
for sentence in examples:
    doc = nlp(sentence)
    print("\\nTEXT:", sentence)
    print("ENTITIES:", [(ent.text, ent.label_) for ent in doc.ents])

\nTEXT: Microsoft was founded by Bill Gates and Paul Allen.
ENTITIES: [('Microsoft', 'ORG'), ('Bill Gates', 'PERSON'), ('Paul Allen', 'PERSON')]
\nTEXT: The World Health Organization is based in Geneva.
ENTITIES: [('The World Health Organization', 'ORG'), ('Geneva', 'GPE')]
\nTEXT: Riya moved from Mumbai to London in June 2024.
ENTITIES: [('Riya', 'PERSON'), ('Mumbai', 'GPE'), ('London', 'GPE'), ('June 2024', 'DATE')]
\nTEXT: The meeting starts at 9:30 AM on Monday.
ENTITIES: [('9:30 AM', 'TIME'), ('Monday', 'DATE')]
\nTEXT: Tesla announced a new product in Germany.
ENTITIES: [('Tesla', 'ORG'), ('Germany', 'GPE')]


## 11. Inspect token-level BIO information

spaCy exposes token-level entity information:
- `token.ent_iob_`: usually `B`, `I`, or `O`.
- `token.ent_type_`: entity category, if any.

In [ ]:
sentence = "Barack Obama visited New Delhi."
doc = nlp(sentence)
for token in doc:
    print(f"{token.text:12} IOB={token.ent_iob_:2} ENTITY_TYPE={token.ent_type_}")

Barack       IOB=B  ENTITY_TYPE=PERSON
Obama        IOB=I  ENTITY_TYPE=PERSON
visited      IOB=O  ENTITY_TYPE=
New          IOB=B  ENTITY_TYPE=GPE
Delhi        IOB=I  ENTITY_TYPE=GPE
.            IOB=O  ENTITY_TYPE=


## 12. Why train a custom NER model?

A pretrained model has a fixed set of learned categories. A university might need custom labels such as `STUDENT`, `COURSE`, and `UNIVERSITY`. We need annotated examples for these categories.

Annotation records the exact character span and its label:

\[
(start,\ end,\ label)
\]

The model learns from these human-provided examples.

## 13. Prepare a small custom dataset

We will build a tiny illustrative university-domain dataset with labels:
- `PERSON`
- `ORG`
- `COURSE`

**Limitation:** The data is synthetic and very small. It demonstrates the API and is not suitable for measuring real-world performance.

In [ ]:
TRAIN_DATA = [
    ("Riya studies NLP at IIIT Allahabad.", {"entities": [(0, 4, "PERSON"), (14, 17, "COURSE"), (21, 35, "ORG")]}),
    ("Aman teaches Machine Learning at ABC University.", {"entities": [(0, 4, "PERSON"), (13, 29, "COURSE"), (33, 47, "ORG")]}),
    ("Neha joined Data Science at XYZ Institute.", {"entities": [(0, 4, "PERSON"), (12, 24, "COURSE"), (28, 41, "ORG")]}),
    ("Rahul studies Computer Vision at ABC University.", {"entities": [(0, 5, "PERSON"), (13, 28, "COURSE"), (32, 46, "ORG")]}),
    ("Priya teaches NLP at XYZ Institute.", {"entities": [(0, 5, "PERSON"), (13, 16, "COURSE"), (20, 33, "ORG")]}),
    ("Karan studies Robotics at IIIT Delhi.", {"entities": [(0, 5, "PERSON"), (14, 22, "COURSE"), (26, 36, "ORG")]}),
    ("Meera teaches Statistics at ABC University.", {"entities": [(0, 5, "PERSON"), (13, 23, "COURSE"), (27, 41, "ORG")]}),
    ("Arjun joined Deep Learning at XYZ Institute.", {"entities": [(0, 5, "PERSON"), (12, 25, "COURSE"), (29, 42, "ORG")]}),
    ("Sara studies NLP at IIIT Delhi.", {"entities": [(0, 4, "PERSON"), (13, 16, "COURSE"), (20, 30, "ORG")]}),
    ("Vikram teaches Robotics at IIIT Allahabad.", {"entities": [(0, 6, "PERSON"), (14, 22, "COURSE"), (26, 41, "ORG")]}),
    ("Anita studies Data Science at ABC University.", {"entities": [(0, 5, "PERSON"), (13, 25, "COURSE"), (29, 43, "ORG")]}),
    ("Dev teaches Computer Vision at XYZ Institute.", {"entities": [(0, 3, "PERSON"), (10, 25, "COURSE"), (29, 42, "ORG")]}),
]
TEST_DATA = [
    ("Asha studies NLP at ABC University.", {"entities": [(0, 4, "PERSON"), (13, 16, "COURSE"), (20, 34, "ORG")]}),
    ("Mohan teaches Robotics at IIIT Delhi.", {"entities": [(0, 5, "PERSON"), (13, 21, "COURSE"), (25, 35, "ORG")]}),
    ("Isha joined Machine Learning at XYZ Institute.", {"entities": [(0, 4, "PERSON"), (12, 28, "COURSE"), (32, 45, "ORG")]}),
]

## 14. Validate annotation offsets
Incorrect character offsets give the model incorrect supervision. This helper prints the spans and checks that they are valid.

In [ ]:
def inspect_annotations(data):
    for text, annotation in data:
        print("\\n", text)
        for start, end, label in annotation["entities"]:
            print(f"{text[start:end]!r:25} -> {label:10} [{start}, {end})")

inspect_annotations(TRAIN_DATA[:3])

def validate_offsets(data):
    for text, annotation in data:
        for start, end, label in annotation["entities"]:
            assert 0 <= start < end <= len(text), (text, start, end, label)
            assert text[start:end].strip() != "", (text, start, end, label)

validate_offsets(TRAIN_DATA)
validate_offsets(TEST_DATA)
print("All annotation offsets are valid.")

\n Riya studies NLP at IIIT Allahabad.
'Riya'                    -> PERSON     [0, 4)
'LP '                     -> COURSE     [14, 17)
'IIT Allahabad.'          -> ORG        [21, 35)
\n Aman teaches Machine Learning at ABC University.
'Aman'                    -> PERSON     [0, 4)
'Machine Learning'        -> COURSE     [13, 29)
'ABC University'          -> ORG        [33, 47)
\n Neha joined Data Science at XYZ Institute.
'Neha'                    -> PERSON     [0, 4)
'Data Science'            -> COURSE     [12, 24)
'XYZ Institute'           -> ORG        [28, 41)
All annotation offsets are valid.


## 15. Create a blank spaCy NER model
A blank pipeline has no pretrained NER knowledge. We add an NER component and register the labels it must learn.

In [ ]:
custom_nlp = spacy.blank("en")
ner = custom_nlp.add_pipe("ner")

LABELS = ["PERSON", "ORG", "COURSE"]
for label in LABELS:
    ner.add_label(label)

print("Pipeline:", custom_nlp.pipe_names)
print("Labels:", ner.labels)

Pipeline: ['ner']
Labels: ('COURSE', 'ORG', 'PERSON')


## 16. Convert annotations into spaCy training examples

An `Example` pairs a predicted document with a reference document containing the correct annotations. `Example.from_dict()` converts our character-span annotations into spaCy's training format.

In [ ]:
from spacy.training import Example

def make_examples(nlp, data):
    examples = []
    for text, annotation in data:
        doc = nlp.make_doc(text)
        examples.append(Example.from_dict(doc, annotation))
    return examples

train_examples = make_examples(custom_nlp, TRAIN_DATA)
test_examples = make_examples(custom_nlp, TEST_DATA)
print("Training examples:", len(train_examples))
print("Test examples:", len(test_examples))

Training examples: 12
Test examples: 3


/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Riya studies NLP at IIIT Allahabad." with entities "[(0, 4, 'PERSON'), (14, 17, 'COURSE'), (21, 35, 'O...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Rahul studies Computer Vision at ABC University." with entities "[(0, 5, 'PERSON'), (13, 28, 'COURSE'), (32, 46, 'O...". Use `spacy.training.offsets_to_biluo_tags(nlp.make_doc(text), entities)` to check the alignment. Misaligned entities ('-') will be ignored during training.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/spacy/training/iob_utils.py:149: UserWarning: [W030] Some entities could not be aligned in the text "Pr

## 17. Train the custom NER model

- `initialize()` initializes model parameters and prepares the pipeline.
- `update()` performs learning updates using annotated examples.
- `losses` stores training loss for the epoch.
- `drop` applies dropout during training.

We shuffle examples at each epoch.

In [ ]:
import random

optimizer = custom_nlp.initialize(get_examples=lambda: train_examples)
N_EPOCHS = 30

for epoch in range(N_EPOCHS):
    random.shuffle(train_examples)
    losses = {}
    custom_nlp.update(
        train_examples,
        sgd=optimizer,
        drop=0.2,
        losses=losses
    )
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch + 1:02d} | Loss: {losses.get('ner', 0):.4f}")

Epoch 05 | Loss: 54.1385
Epoch 10 | Loss: 41.2825
Epoch 15 | Loss: 21.8025
Epoch 20 | Loss: 13.5146
Epoch 25 | Loss: 29.2935
Epoch 30 | Loss: 41.3007


### Understand training loss

A decreasing training loss often means the model is fitting the training examples. It does not guarantee generalization. A tiny dataset can be memorized, so evaluation on unseen, representative examples is essential.

## 18. Test the custom model

In [ ]:
test_sentence = "Asha studies NLP at ABC University."
doc = custom_nlp(test_sentence)
print("Text:", test_sentence)
print("Entities:", [(ent.text, ent.label_) for ent in doc.ents])
displacy.render(doc, style="ent", jupyter=True)

Text: Asha studies NLP at ABC University.
Entities: [('Asha', 'PERSON'), ('NLP', 'COURSE'), ('ABC University.', 'ORG')]


In [ ]:
new_sentences = [
    "Mohan studies Robotics at IIIT Delhi.",
    "Isha teaches Machine Learning at XYZ Institute.",
    "Karan studies NLP at ABC University."
]
for sentence in new_sentences:
    doc = custom_nlp(sentence)
    print(sentence)
    print([(ent.text, ent.label_) for ent in doc.ents])
    print()

Mohan studies Robotics at IIIT Delhi.
[('Mohan', 'PERSON'), ('Robotics', 'COURSE'), ('IIIT Delhi.', 'ORG')]

Isha teaches Machine Learning at XYZ Institute.
[('Isha', 'PERSON'), ('Machine Learning', 'COURSE'), ('XYZ Institute.', 'ORG')]

Karan studies NLP at ABC University.
[('Karan', 'PERSON'), ('NLP', 'COURSE'), ('ABC University.', 'ORG')]



## 19. Evaluate using entity-level precision, recall, and F1

A prediction is correct only when its start, end, and label all exactly match a gold entity.

- TP: exact predicted entity match.
- FP: predicted entity not present in gold annotations.
- FN: gold entity missed by the model.

\[
Precision=\frac{TP}{TP+FP}
\qquad
Recall=\frac{TP}{TP+FN}
\]

\[
F1=2\cdot\frac{Precision\cdot Recall}{Precision+Recall}
\]

Precision asks: of predicted entities, how many were correct? Recall asks: of gold entities, how many were found?

In [ ]:
def get_gold_entities(annotation):
    return {(start, end, label) for start, end, label in annotation["entities"]}

def get_predicted_entities(doc):
    return {(ent.start_char, ent.end_char, ent.label_) for ent in doc.ents}

tp = fp = fn = 0
for text, annotation in TEST_DATA:
    doc = custom_nlp(text)
    gold = get_gold_entities(annotation)
    predicted = get_predicted_entities(doc)
    tp += len(gold & predicted)
    fp += len(predicted - gold)
    fn += len(gold - predicted)

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0
f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

print(f"TP: {tp}, FP: {fp}, FN: {fn}")
print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")

TP: 5, FP: 4, FN: 4
Precision: 0.556
Recall:    0.556
F1-score:  0.556


**Evaluation caution:** The test set has only three synthetic sentences. Its scores are not reliable estimates of real-world performance. Use a sufficiently large, representative, independently annotated test set.

## 20. Save and reload the model

In [ ]:
output_dir = "./custom_university_ner"
custom_nlp.to_disk(output_dir)
print("Model saved to:", output_dir)

Model saved to: ./custom_university_ner


In [ ]:
reloaded_nlp = spacy.load(output_dir)
text = "Riya studies NLP at IIIT Allahabad."
doc = reloaded_nlp(text)
print([(ent.text, ent.label_) for ent in doc.ents])

[('Riya', 'PERSON'), ('NLP', 'COURSE'), ('IIIT Allahabad.', 'ORG')]


## 21. Common mistakes and practical advice

1. Incorrect character offsets: verify `text[start:end]`.
2. Inconsistent labels: define a label schema and follow it.
3. Too little data: a dozen examples only demonstrate the API.
4. Data leakage: do not evaluate on training examples.
5. Incorrect boundaries: predicting `New` instead of `New Delhi` is an entity-level error.
6. Capitalization alone is not enough to identify entities.
7. Annotation guidelines should clarify titles, abbreviations, partial names, and nested entities.
8. Domain mismatch can make a general model unreliable on clinical, legal, or financial text.

Production work needs larger datasets, clear annotation guidelines, representative test data, and error analysis.

## 22. Classroom exercises

**Exercise 1:** Manually annotate the entities in: `Dr. A.P.J. Abdul Kalam worked at DRDO in Hyderabad.`

**Exercise 2:** Try sentences containing people, organizations, locations, dates, and money with the pretrained model.

**Exercise 3:** Add a custom label such as `DEPARTMENT`. Annotate at least five new training examples, then reinitialize and retrain.

**Exercise 4:** Inspect an incorrect prediction. Is it a boundary error, wrong category, missed entity, or false positive?

**Exercise 5:** For healthcare NER, propose labels such as `DISEASE`, `MEDICINE`, and `SYMPTOM`. What annotation rules and privacy safeguards are needed?

## 23. Key takeaways

- NER identifies entity spans and assigns categories.
- It is commonly formulated as sequence labeling.
- BIO tags represent beginning, inside, and outside positions.
- Pretrained models are useful, but their label sets and domain coverage are limited.
- Custom NER requires correctly annotated examples.
- spaCy supports inference, training, evaluation, saving, and reloading.
- Entity-level precision, recall, and F1 are useful evaluation measures.

**Suggested classroom sequence:** Sections 1–5 for theory, 6–11 for pretrained NER, and 12–20 for custom model training and evaluation.